In [ ]:
!pip install langchain==1.2.15 langchain-community langchain-core==1.2.28 langchain-ollama==1.1.0

In [ ]:
!pip install -U ollama

- ollama pull gpt-oss:20b

In [ ]:
!sudo apt update
!sudo apt install -y pciutils
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
import threading
import subprocess
import time

def run_ollama_serve():
  subprocess.Popen(["ollama", "serve"])

thread = threading.Thread(target=run_ollama_serve)
thread.start()
time.sleep(5)

這一步需要花一些時間


In [ ]:
!ollama pull gpt-oss:20b

# Ollama

趁著上面在下載的時候去逛逛人家的網頁

https://ollama.com/

重點是知道如何調用API，剩下的部分跟使用OpenAI API的情況下一樣。

## Ollama 本地部屬

In [ ]:
from langchain_ollama import ChatOllama

model_id = "gpt-oss:20b"

model_gpt_oss_20b = ChatOllama(model=model_id, temperature=0)

In [ ]:
model_gpt_oss_20b.invoke("How are you today?")

一個20b的模型就需要12G GRAM

一個70b的模型可能也就剛好過可以跑Agent的門檻
就算使用Multi-Agent的架構，還是需要一個非常強大的模型來Orchestra其他的小模型，所以有些東西還是跑不掉的。架設小的語言模型也還是相當吃硬體資源的。

卸載模型

In [ ]:
!curl http://localhost:11434/api/generate -d '{"model": "gpt-oss:20b", "keep_alive": 0}'

In [ ]:
!ollama pull gemma4:e2b

In [ ]:
model_id = "gemma4:e2b"

model_gemma = ChatOllama(model=model_id, temperature=0)

In [ ]:
human_message = HumanMessage(content="How are you today?")

model_gemma.invoke([human_message])

In [ ]:
import base64

with open("bottle-detection.mp4", "rb") as video_file:
  video_data = base64.b64encode(video_file.read()).decode('utf-8')

目前Langchain還沒支援Video作為輸入，所以需要將Video轉換成一系列的圖像。這個技巧應該是對於所有支援圖像的大語言模型都適用

In [ ]:
import cv2
import base64
from langchain_core.prompts import ChatPromptTemplate, SystemMessagePromptTemplate, HumanMessagePromptTemplate, PromptTemplate
from langchain_core.messages import SystemMessage, HumanMessage


import base64
from IPython.display import display, clear_output
from PIL import Image
import time

def extract_and_show_frames(video_path, num_frames=5):
    video = cv2.VideoCapture(video_path)
    total_frames = int(video.get(cv2.CAP_PROP_FRAME_COUNT))
    interval = total_frames // num_frames

    base64_frames = []
    
    for i in range(num_frames):
        video.set(cv2.CAP_PROP_POS_FRAMES, i * interval)
        success, frame = video.read()
        
        if success:
            # 1. 將 OpenCV 的 BGR 格式轉為 RGB 格式 (PIL 需要)
            rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            img = Image.fromarray(rgb_frame)
            
            # 2. 清除當前儲存格輸出，並顯示新圖片
            clear_output(wait=True)
            print(f"正在抽取第 {i+1}/{num_frames} 張圖片...")
            display(img)
            
            # 3. 轉為 Base64 供後續使用
            _, buffer = cv2.imencode(".jpg", frame)
            base64_frames.append(base64.b64encode(buffer).decode("utf-8"))
            
            # 稍微停頓一下，讓你肉眼能看清楚
            time.sleep(0.5) 

    video.release()
    print("抽取完成！")
    return base64_frames

# 執行函式
frames = extract_and_show_frames("bottle-detection.mp4", num_frames=5)

# 2. Build the Multi-modal content list
content = [{"type": "text", "text": "What is happening in this video?"}]

for frame_data in frames:
    content.append({
        "type": "image_url",
        "image_url": {"url": f"data:image/jpeg;base64,{frame_data}"}
    })

# 3. Send to LangChain
human_message = HumanMessage(content=content)

In [ ]:
model_gemma.invoke([human_message])